# Assignment 1: Vector Database Creation and Retrieval
## Day 6 Session 2 - RAG Fundamentals

**OBJECTIVE:** Create a vector database from a folder of documents and implement basic retrieval functionality.

**LEARNING GOALS:**
- Understand document loading with SimpleDirectoryReader
- Learn vector store setup with LanceDB
- Implement vector index creation
- Perform semantic search and retrieval

**DATASET:** Use the data folder in `Day_6/session_2/data/` which contains multiple file types

**INSTRUCTIONS:**
1. Complete each function by replacing the TODO comments with actual implementation
2. Run each cell after completing the function to test it
3. The answers can be found in the existing notebooks in the `llamaindex_rag/` folder


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone https://github.com/eng-accelerator/ai-accelerator-C5.git

Cloning into 'ai-accelerator-C5'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 145 (delta 11), reused 4 (delta 4), pack-reused 127 (from 2)
Receiving objects: 100% (145/145), 26.04 MiB | 21.11 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [4]:
!pip install -r "/content/ai-accelerator-C5/Day_6/session_2/requirements.txt"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 16.4 MB/s eta 0:00:00
  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-index-llms-openai-like to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of llama-index-

In [1]:
# Import required libraries
import os
from pathlib import Path
from typing import List
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, StorageContext, Settings
from llama_index.vector_stores.lancedb import LanceDBVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [5]:
# Configure LlamaIndex Settings (Using OpenRouter - No OpenAI API Key needed)
from google.colab import userdata

def setup_llamaindex_settings():
    """
    Configure LlamaIndex with local embeddings and OpenRouter for LLM.
    This assignment focuses on vector database operations, so we'll use local embeddings only.
    """
    # Check for OpenRouter API key (for future use, not needed for this basic assignment)
    #api_key = os.getenv("OPENROUTER_API_KEY")
    api_key = userdata.get('OPENROUTER_API_KEY')
    if not api_key:
        print("ℹ️  OPENROUTER_API_KEY not found - that's OK for this assignment!")
        print("   This assignment only uses local embeddings for vector operations.")

    # Configure local embeddings (no API key required)
    Settings.embed_model = HuggingFaceEmbedding(
        model_name="BAAI/bge-small-en-v1.5",
        trust_remote_code=True
    )

    print("✅ LlamaIndex configured with local embeddings")
    print("   Using BAAI/bge-small-en-v1.5 for document embeddings")

# Setup the configuration
setup_llamaindex_settings()


✅ LlamaIndex configured with local embeddings
   Using BAAI/bge-small-en-v1.5 for document embeddings


## 1. Document Loading Function

Complete the function below to load documents from a folder using `SimpleDirectoryReader`.

**Note:** This assignment uses local embeddings only - no OpenAI API key required! We're configured to use OpenRouter for future LLM operations.


In [8]:
# def load_documents_from_folder(folder_path: str):
#     """
#     Load documents from a folder using SimpleDirectoryReader.

#     TODO: Complete this function to load documents from the given folder path.
#     HINT: Use SimpleDirectoryReader with recursive parameter to load all files

#     Args:
#         folder_path (str): Path to the folder containing documents

#     Returns:
#         List of documents loaded from the folder
#     """
#     # TODO: Create SimpleDirectoryReader instance
#     # reader = ?

#     # TODO: Load and return documents
#     # documents = ?

#     # return documents

#     # PLACEHOLDER - Replace with actual implementation
#     print(f"TODO: Load documents from {folder_path}")
#     return []

# # Test the function after you complete it
# test_folder = "/content/ai-accelerator-C5/Day_6/session_2/data"
# documents = load_documents_from_folder(test_folder)
# print(f"Loaded {len(documents)} documents")

from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(folder_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    Args:
        folder_path (str): Path to the folder containing documents

    Returns:
        List of documents loaded from the folder
    """
    reader = SimpleDirectoryReader(
        input_dir=folder_path,
        recursive=True          # Traverse sub-folders too
    )

    documents = reader.load_data()

    return documents

# Test the function
test_folder = "/content/ai-accelerator-C5/Day_6/session_2/data"
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 186MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Loaded 42 documents


## 2. Vector Store Creation Function

Complete the function below to create a LanceDB vector store.


In [10]:
# def create_vector_store(db_path: str = "./vectordb", table_name: str = "documents"):
#     """
#     Create a LanceDB vector store for storing document embeddings.

#     TODO: Complete this function to create and configure a LanceDB vector store.
#     HINT: Use LanceDBVectorStore with uri and table_name parameters

#     Args:
#         db_path (str): Path where the vector database will be stored
#         table_name (str): Name of the table in the vector database

#     Returns:
#         LanceDBVectorStore: Configured vector store
#     """
#     # TODO: Create the directory if it doesn't exist
#     # Path(db_path).mkdir(parents=True, exist_ok=True)

#     # TODO: Create vector store
#     # vector_store = ?

#     # return vector_store

#     # PLACEHOLDER - Replace with actual implementation
#     print(f"TODO: Create vector store at {db_path}")
#     return None

# # Test the function after you complete it
# vector_store = create_vector_store("./assignment_vectordb")
# print(f"Vector store created: {vector_store is not None}")
from pathlib import Path
from llama_index.vector_stores.lancedb import LanceDBVectorStore

def create_vector_store(db_path: str = "./vectordb", table_name: str = "documents"):
    """
    Create a LanceDB vector store for storing document embeddings.

    Args:
        db_path (str): Path where the vector database will be stored
        table_name (str): Name of the table in the vector database

    Returns:
        LanceDBVectorStore: Configured vector store
    """
    # Ensure the directory exists
    Path(db_path).mkdir(parents=True, exist_ok=True)

    # Create and return the LanceDB vector store
    vector_store = LanceDBVectorStore(
        uri=db_path,
        table_name=table_name
    )

    return vector_store

# Test the function
vector_store = create_vector_store("./assignment_vectordb")
print(f"Vector store created: {vector_store is not None}")

Vector store created: True


## 3. Vector Index Creation Function

Complete the function below to create a vector index from documents.


In [11]:
# def create_vector_index(documents: List, vector_store):
#     """
#     Create a vector index from documents using the provided vector store.

#     TODO: Complete this function to create a VectorStoreIndex from documents.
#     HINT: Create StorageContext with vector_store, then use VectorStoreIndex.from_documents()

#     Args:
#         documents: List of documents to index
#         vector_store: LanceDB vector store to use for storage

#     Returns:
#         VectorStoreIndex: The created vector index
#     """
#     # TODO: Create storage context with vector store
#     # storage_context = ?

#     # TODO: Create index from documents
#     # index = ?

#     # return index

#     # PLACEHOLDER - Replace with actual implementation
#     print(f"TODO: Create vector index from {len(documents)} documents")
#     return None

# # Test the function after you complete it (will only work after previous functions are completed)
# if documents and vector_store:
#     index = create_vector_index(documents, vector_store)
#     print(f"Vector index created: {index is not None}")
# else:
#     print("Complete previous functions first to test this one")
from typing import List
from llama_index.core import VectorStoreIndex, StorageContext

def create_vector_index(documents: List, vector_store):
    """
    Create a vector index from documents using the provided vector store.

    Args:
        documents: List of documents to index
        vector_store: LanceDB vector store to use for storage

    Returns:
        VectorStoreIndex: The created vector index
    """
    # Wrap the vector store in a StorageContext
    storage_context = StorageContext.from_defaults(
        vector_store=vector_store
    )

    # Build the index — this chunks, embeds, and stores all documents
    index = VectorStoreIndex.from_documents(
        documents,
        storage_context=storage_context,
        show_progress=True      # Progress bar during embedding
    )

    return index

# Test the function
if documents and vector_store:
    index = create_vector_index(documents, vector_store)
    print(f"Vector index created: {index is not None}")
else:
    print("Complete previous functions first to test this one")

Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/55 [00:00<?, ?it/s]

Vector index created: True


## 4. Document Search Function

Complete the function below to search for relevant documents using the vector index.


In [12]:
# def search_documents(index, query: str, top_k: int = 3):
#     """
#     Search for relevant documents using the vector index.

#     TODO: Complete this function to perform semantic search on the index.
#     HINT: Use index.as_retriever() with similarity_top_k parameter, then retrieve(query)

#     Args:
#         index: Vector index to search
#         query (str): Search query
#         top_k (int): Number of top results to return

#     Returns:
#         List of retrieved document nodes
#     """
#     # TODO: Create retriever from index
#     # retriever = ?

#     # TODO: Retrieve documents for the query
#     # results = ?

#     # return results

#     # PLACEHOLDER - Replace with actual implementation
#     print(f"TODO: Search for '{query}' in index")
#     return []

# # Test the function after you complete it (will only work after all previous functions are completed)
# if 'index' in locals() and index is not None:
#     test_query = "What are AI agents?"
#     results = search_documents(index, test_query, top_k=2)
#     print(f"Found {len(results)} results for query: '{test_query}'")
#     for i, result in enumerate(results, 1):
#         print(f"Result {i}: {result.text[:100] if hasattr(result, 'text') else 'No text'}...")
# else:
#     print("Complete all previous functions first to test this one")
def search_documents(index, query: str, top_k: int = 3):
    """
    Search for relevant documents using the vector index.

    Args:
        index: Vector index to search
        query (str): Search query
        top_k (int): Number of top results to return

    Returns:
        List of retrieved document nodes
    """
    # Create a retriever with the desired top-k
    retriever = index.as_retriever(
        similarity_top_k=top_k
    )

    # Retrieve the most relevant nodes for the query
    results = retriever.retrieve(query)

    return results

# Test the function
if 'index' in locals() and index is not None:
    test_query = "What are AI agents?"
    results = search_documents(index, test_query, top_k=2)
    print(f"Found {len(results)} results for query: '{test_query}'")
    for i, result in enumerate(results, 1):
        print(f"Result {i}: {result.text[:100] if hasattr(result, 'text') else 'No text'}...")
else:
    print("Complete all previous functions first to test this one")

Found 2 results for query: 'What are AI agents?'
Result 1: task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m...
Result 2: agent-personas or the user is not needed, multi-agent architectures tend to thrive more when collabo...


## 5. Final Test - Complete Pipeline

Once you've completed all the functions above, run this cell to test the complete pipeline with multiple search queries.


In [14]:
# Final test of the complete pipeline
print("🚀 Testing Complete Vector Database Pipeline")
print("=" * 50)

# Re-run the complete pipeline to ensure everything works
#data_folder = "../data"
data_folder = "/content/ai-accelerator-C5/Day_6/session_2/data"
vector_db_path = "./assignment_vectordb"

# Step 1: Load documents
print("\n📂 Step 1: Loading documents...")
documents = load_documents_from_folder(data_folder)
print(f"   Loaded {len(documents)} documents")

# Step 2: Create vector store
print("\n🗄️ Step 2: Creating vector store...")
vector_store = create_vector_store(vector_db_path)
print("   Vector store status:", "✅ Created" if vector_store else "❌ Failed")

# Step 3: Create vector index
print("\n🔗 Step 3: Creating vector index...")
if documents and vector_store:
    index = create_vector_index(documents, vector_store)
    print("   Index status:", "✅ Created" if index else "❌ Failed")
else:
    index = None
    print("   ❌ Cannot create index - missing documents or vector store")

# Step 4: Test multiple search queries
print("\n🔍 Step 4: Testing search functionality...")
if index:
    search_queries = [
        "What are AI agents?",
        "How to evaluate agent performance?",
        "Italian recipes and cooking",
        "Financial analysis and investment"
    ]

    for query in search_queries:
        print(f"\n   🔎 Query: '{query}'")
        results = search_documents(index, query, top_k=2)

        if results:
            for i, result in enumerate(results, 1):
                text_preview = result.text[:100] if hasattr(result, 'text') else "No text available"
                score = f" (Score: {result.score:.4f})" if hasattr(result, 'score') else ""
                print(f"      {i}. {text_preview}...{score}")
        else:
            print("      No results found")
else:
    print("   ❌ Cannot test search - index not created")

print("\n" + "=" * 50)
print("🎯 Assignment Status:")
print(f"   Documents loaded: {'✅' if documents else '❌'}")
print(f"   Vector store created: {'✅' if vector_store else '❌'}")
print(f"   Index created: {'✅' if index else '❌'}")
print(f"   Search working: {'✅' if index else '❌'}")

if documents and vector_store and index:
    print("\n🎉 Congratulations! You've successfully completed the assignment!")
    print("   You've built a complete vector database with search functionality!")
else:
    print("\n📝 Please complete the TODO functions above to finish the assignment.")


🚀 Testing Complete Vector Database Pipeline

📂 Step 1: Loading documents...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


   Loaded 42 documents

🗄️ Step 2: Creating vector store...
   Vector store status: ✅ Created

🔗 Step 3: Creating vector index...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/55 [00:00<?, ?it/s]

   Index status: ✅ Created

🔍 Step 4: Testing search functionality...

   🔎 Query: 'What are AI agents?'
      1. task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m... (Score: 0.6198)
      2. task. In single agent patterns there is no feedback mechanism from other AI agents; however, there m... (Score: 0.6198)

   🔎 Query: 'How to evaluate agent performance?'
      1. steps, but the answers are limited to Yes/No responses [7]. As the industry continues to pivot towar... (Score: 0.6783)
      2. steps, but the answers are limited to Yes/No responses [7]. As the industry continues to pivot towar... (Score: 0.6783)

   🔎 Query: 'Italian recipes and cooking'
      1. Spaghetti Carbonara, Italian, 20, Easy, Pasta, 450
Margherita Pizza, Italian, 45, Medium, Tomato, 32... (Score: 0.6246)
      2. Spaghetti Carbonara, Italian, 20, Easy, Pasta, 450
Margherita Pizza, Italian, 45, Medium, Tomato, 32... (Score: 0.6246)

   🔎 Query: 'Financial anal

In [ ]:
# def load_documents_from_folder(folder_path: str):
#     """
#     Load documents from a folder using SimpleDirectoryReader.

#     TODO: Complete this function to load documents from the given folder path.
#     HINT: Use SimpleDirectoryReader with recursive parameter to load all files

#     Args:
#         folder_path (str): Path to the folder containing documents

#     Returns:
#         List of documents loaded from the folder
#     """
#     # TODO: Create SimpleDirectoryReader instance
#     # reader = ?

#     # TODO: Load and return documents
#     # documents = ?

#     # return documents

#     # PLACEHOLDER - Replace with actual implementation
#     print(f"TODO: Load documents from {folder_path}")
#     return []

# # Test the function after you complete it
# test_folder = "/content/ai-accelerator-C5/Day_6/session_2/data"
# documents = load_documents_from_folder(test_folder)
# print(f"Loaded {len(documents)} documents")

from llama_index.core import SimpleDirectoryReader

def load_documents_from_folder(folder_path: str):
    """
    Load documents from a folder using SimpleDirectoryReader.

    Args:
        folder_path (str): Path to the folder containing documents

    Returns:
        List of documents loaded from the folder
    """
    reader = SimpleDirectoryReader(
        input_dir=folder_path,
        recursive=True          # Traverse sub-folders too
    )

    documents = reader.load_data()

    return documents

# Test the function
test_folder = "/content/ai-accelerator-C5/Day_6/session_2/data"
documents = load_documents_from_folder(test_folder)
print(f"Loaded {len(documents)} documents")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 186MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Loaded 42 documents
